# Button Press Timings from UPPT002

This notebook loads each participant's MEG epochs and detects the initial UPPT002 signal changes that follow a button press. Detected times are matched to the behavioral press times in order so they can be placed into the correct press slot.

The saved array has the dimensions (participant, trial, press slot). Times are in seconds relative to cue onset. Missing detections and trials not recorded for a participant are stored as NaN. Subject 16 has 222 recorded trials; other participants are expected to have 240.

import packages for loading the behavioral files, MEG epochs and plotting the quality checks.

In [ ]:
import os
import h5py
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

choose the participants to process and set the transition detection values

In [ ]:
sub_ID = ['05','06','08','09','10','11','13','14','15','16','17','18','20','21','24','25','28']  # subject IDs
excluded_subjects = []                      # participants with questionable data to exclude

n_press_slots = 5                           # number of button presses in each trial
CUE_ONSET_AFTER_EPOCH_START_S = 2.8         # cue onset after the start of each MEG epoch
MATCH_TOLERANCE_S = 0.25                    # maximum difference used to match a signal change to a behavioral press

# Signal changes must pass the adaptive step threshold and remain present for 20 ms.
NOISE_SIGMA_MULTIPLIER = 8.0
RELATIVE_STEP_FRACTION = 0.02
PERSISTENCE_S = 0.020
REFRACTORY_S = 0.020

load behavioral press times and MEG epochs for each participant

The file paths and subject naming follow MEG_powerplot_all.ipynb. Behavioral times are converted from milliseconds to seconds and include the cue duration. Subject 16 has 222 MEG epochs, so its first 222 behavioral rows are used.

In [ ]:
def filename_loader(id):
    # id is the subject number as a two-digit string.
    subject_folder = "//rdp.arc.ucl.ac.uk/ritd-ag-project-rd01pz-dbush99/Katja Button Press Data"
    button_press_folder = "/beh_data/s" + str(id)
    BP1 = subject_folder + button_press_folder + "/behDataMEG.mat"
    BP2 = subject_folder + button_press_folder + "/targetMEG.mat"

    # Subject folder names use a leading zero for single-digit IDs.
    if id.startswith("0"):
        spliced_id = id[1:]
        MEG_file = "/s0" + spliced_id + "/Subject" + spliced_id + "_spm_mne-epo"
    else:
        MEG_file = "/s" + str(id) + "/Subject" + str(id) + "_spm_mne-epo"
    epochs_file = subject_folder + MEG_file + ".fif"

    # Load the behavioral press times.
    with h5py.File(BP1, "r") as f:
        timings = f["A/timing"][:].T

    # Load the duration of each trial's cue.
    with h5py.File(BP2, "r") as f:
        cue_duration = np.asarray(f["T/cueDur"][:]).reshape(-1)

    if timings.shape[0] != len(cue_duration):
        raise ValueError(
            "Subject " + id + " has different trial counts in A/timing and T/cueDur"
        )

    # Add cue duration to each press time and convert milliseconds to seconds.
    press_times = (timings + cue_duration[:, None]) / 1000.0

    # Load the MEG epochs.
    epochs = mne.read_epochs(epochs_file, preload=True)

    if "UPPT002" not in epochs.ch_names:
        raise ValueError("UPPT002 is missing from the MEG epochs for subject " + id)

    # Keep the first 222 behavioral rows for subject 16.
    if id == "16" and len(epochs) == 222 and press_times.shape[0] == 240:
        print("Subject 16: using the first 222 behavioral trial rows.")
        press_times = press_times[:len(epochs), :]

    # Check that each MEG epoch has a behavioral trial.
    elif len(epochs) != press_times.shape[0]:
        raise ValueError(
            "Subject " + id + " has " + str(len(epochs))
            + " MEG epochs but " + str(press_times.shape[0]) + " behavioral trials"
        )

    return press_times, epochs

select the participants that will be processed

All IDs in sub_ID are processed except IDs in excluded_subjects.

In [ ]:
subjects_to_use = [id for id in sub_ID if id not in excluded_subjects]

if len(subjects_to_use) == 0:
    raise ValueError("No subjects remain after applying excluded_subjects.")

print("Subjects to process:", subjects_to_use)

detect the initial changes from the UPPT002 baseline

The baseline is estimated with its median and median absolute deviation. A candidate change must exceed both the noise-based and signal-relative step thresholds, persist for 20 ms, and move away from the baseline level. The 20 ms refractory period prevents a single transition from being counted more than once.

In [ ]:
def robust_sigma(values):
    # Estimate spread with the median absolute deviation.
    center = np.median(values)
    return 1.4826 * np.median(np.abs(values - center))


def detect_uppt002_changes(signal, epoch_times, cue_onset):
    # Use the pre-cue signal to estimate the UPPT002 baseline.
    baseline_window = (
        (epoch_times >= epoch_times[0] + 0.05)
        & (epoch_times < cue_onset - 0.05)
    )
    baseline_values = signal[baseline_window]

    if len(baseline_values) < 3:
        raise ValueError("UPPT002 baseline window is too short to estimate noise")

    baseline = np.median(baseline_values)
    baseline_noise = robust_sigma(baseline_values)
    step_noise = robust_sigma(np.diff(baseline_values))

    excursion = np.percentile(np.abs(signal - baseline), 99.9)
    numerical_floor = np.finfo(float).eps * max(1.0, np.max(np.abs(signal))) * 10
    step_threshold = max(
        NOISE_SIGMA_MULTIPLIER * step_noise,
        RELATIVE_STEP_FRACTION * excursion,
        numerical_floor,
    )
    active_threshold = max(
        3 * baseline_noise, 0.25 * step_threshold, numerical_floor
    )

    sfreq = 1 / np.median(np.diff(epoch_times))
    persistence = max(3, int(np.ceil(PERSISTENCE_S * sfreq)))
    refractory = max(1, int(np.ceil(REFRACTORY_S * sfreq)))
    edges = np.where(
        np.abs(np.diff(signal, prepend=signal[0])) >= step_threshold
    )[0]

    event_samples = []
    last_event = -refractory

    for sample in edges:
        if sample < persistence or sample + persistence > len(signal):
            continue
        if sample - last_event < refractory:
            continue

        before = np.median(signal[sample - persistence:sample])
        after = np.median(signal[sample:sample + persistence])

        if abs(after - before) < step_threshold:
            continue
        if abs(after - baseline) <= active_threshold:
            continue

        event_samples.append(sample)
        last_event = sample

    event_samples = np.asarray(event_samples, dtype=int)

    # Return event times in seconds relative to cue onset.
    event_times = epoch_times[event_samples] - cue_onset

    info = {
        "baseline_noise": baseline_noise,
        "step_threshold": step_threshold,
    }
    return event_times, info

match detected signal changes to behavioral press slots

The matcher keeps the largest possible number of in-order matches within MATCH_TOLERANCE_S. When several matches are possible, it chooses the set with the smallest total time difference. Unmatched slots remain NaN in the saved array.

In [ ]:
def match_events_in_order(behavior_times, uppt_times, tolerance):
    behavior_order = np.flatnonzero(np.isfinite(behavior_times))
    behavior_order = behavior_order[
        np.argsort(behavior_times[behavior_order], kind="stable")
    ]
    uppt_order = np.argsort(uppt_times, kind="stable")

    behavior = behavior_times[behavior_order]
    uppt = uppt_times[uppt_order]
    n_behavior = len(behavior)
    n_uppt = len(uppt)

    count = np.zeros((n_behavior + 1, n_uppt + 1), dtype=int)
    cost = np.zeros((n_behavior + 1, n_uppt + 1))
    step = np.zeros((n_behavior + 1, n_uppt + 1), dtype=int)

    for i in range(1, n_behavior + 1):
        for j in range(1, n_uppt + 1):
            options = [
                (count[i - 1, j], cost[i - 1, j], 1),
                (count[i, j - 1], cost[i, j - 1], 2),
            ]

            difference = abs(behavior[i - 1] - uppt[j - 1])
            if difference <= tolerance:
                options.append(
                    (count[i - 1, j - 1] + 1,
                     cost[i - 1, j - 1] + difference, 3)
                )

            best = max(options, key=lambda x: (x[0], -x[1], x[2] == 3))
            count[i, j], cost[i, j], step[i, j] = best

    pairs = []
    i, j = n_behavior, n_uppt

    while i > 0 and j > 0:
        if step[i, j] == 3:
            pairs.append((behavior_order[i - 1], uppt_order[j - 1]))
            i -= 1
            j -= 1
        elif step[i, j] == 1:
            i -= 1
        else:
            j -= 1

    return pairs[::-1]

process each participant and build a trial-by-press array

For every trial, detected UPPT002 changes are matched to behavioral times. Matched signal-change times are written into the corresponding behavioral press slot; the behavioral time itself is used only for matching. The output times therefore come from UPPT002.

In [ ]:
participant_times = []
subjects_used = []
n_trials_by_subject = []
trial_rows = []

for id in subjects_to_use:
    press_times, epochs = filename_loader(id)

    cue_onset_epoch_s = epochs.times[0] + CUE_ONSET_AFTER_EPOCH_START_S
    uppt002_data = epochs.get_data(picks=["UPPT002"])[:, 0, :]
    times_by_trial = np.full((len(epochs), n_press_slots), np.nan)

    for trial_index in range(len(epochs)):
        behavior_times = press_times[trial_index]
        uppt_times, detector_info = detect_uppt002_changes(
            uppt002_data[trial_index], epochs.times, cue_onset_epoch_s
        )

        pairs = match_events_in_order(
            behavior_times, uppt_times, MATCH_TOLERANCE_S
        )

        # Store each matched UPPT002 time in its behavioral press slot.
        for press_index, uppt_index in pairs:
            times_by_trial[trial_index, press_index] = uppt_times[uppt_index]

        n_behavior = int(np.isfinite(behavior_times).sum())
        trial_rows.append({
            "participant": id,
            "trial": trial_index + 1,
            "n_behavior_expected": n_behavior,
            "n_uppt_detected": len(uppt_times),
            "n_matched": len(pairs),
            "n_behavior_unmatched": n_behavior - len(pairs),
            "n_uppt_unmatched": len(uppt_times) - len(pairs),
            "baseline_noise": detector_info["baseline_noise"],
            "step_threshold": detector_info["step_threshold"],
        })

    participant_times.append(times_by_trial)
    subjects_used.append(id)
    n_trials_by_subject.append(len(epochs))

    print(
        "Subject", id, "- trials:", len(epochs),
        "- matched presses:", int(np.isfinite(times_by_trial).sum())
    )

# Pad shorter participants with NaN so the result is one 3D array.
max_n_trials = max(n_trials_by_subject)
UPPT002_times = np.full(
    (len(subjects_used), max_n_trials, n_press_slots), np.nan
)

for subject_index, times_by_trial in enumerate(participant_times):
    UPPT002_times[subject_index, :len(times_by_trial), :] = times_by_trial

trial_qc_table = pd.DataFrame(trial_rows)

print("UPPT002 array shape:", UPPT002_times.shape)
print("Participant row order:", subjects_used)

check detections across participants and trials

The participant table reports the number of expected behavioral presses and the number of matched UPPT002 changes. The trial table highlights trials with one or more expected presses that did not have a matched signal change.

In [ ]:
summary_rows = []

for id in subjects_used:
    subject_trials = trial_qc_table[trial_qc_table["participant"] == id]
    n_expected = int(subject_trials["n_behavior_expected"].sum())
    n_matched = int(subject_trials["n_matched"].sum())

    summary_rows.append({
        "participant": id,
        "n_trials": len(subject_trials),
        "n_expected_presses": n_expected,
        "n_uppt_detected": int(subject_trials["n_uppt_detected"].sum()),
        "n_matched_presses": n_matched,
        "match_rate": n_matched / n_expected if n_expected else np.nan,
        "n_trials_with_unmatched_presses": int(
            (subject_trials["n_behavior_unmatched"] > 0).sum()
        ),
    })

summary_table = pd.DataFrame(summary_rows)
display(summary_table.round(4))

# Review trials where an expected press did not have a matching UPPT002 change.
incomplete_trials = trial_qc_table[
    trial_qc_table["n_behavior_unmatched"] > 0
]
display(incomplete_trials.head(20))

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(summary_table["participant"], summary_table["match_rate"], color="slateblue")
ax.set_ylim(0, 1)
ax.set_xlabel("Participant")
ax.set_ylabel("Matched presses / expected presses")
ax.set_title("UPPT002 press detection rate")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

save the participant-by-trial-by-press array

The compressed file includes the array, participant row order, each participant's recorded trial count, and press-slot labels. When the notebook is run from the repository folder, the file is saved in that folder.

In [ ]:
OUTPUT_FILE = os.path.join(os.getcwd(), "UPPT002_button_press_times.npz")

np.savez_compressed(
    OUTPUT_FILE,
    uppt002_times=UPPT002_times,
    subject_ID=np.asarray(subjects_used),
    n_trials_by_subject=np.asarray(n_trials_by_subject, dtype=int),
    press_slot=np.arange(1, n_press_slots + 1),
    cue_onset_after_epoch_start_s=CUE_ONSET_AFTER_EPOCH_START_S,
    match_tolerance_s=MATCH_TOLERANCE_S,
    time_reference="seconds relative to cue onset",
)

print("Saved UPPT002 button press times to:", OUTPUT_FILE)